# HyperCLOVA 정책 검색 PPO 학습
**이 파일의 역할:** GPU 서버에서 환경 확인부터 검색, Actor/Critic 로딩, PPO 학습, 평가와 체크포인트 재개까지 실행하는 실험 노트북입니다.

모델: `naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B`  
구조: 질문 → Actor → 검색 → 추가 검색/답변 → 정답 보상 + 독립 Critic → PPO

**먼저 [README.md](README.md)의 서버 설치·SSH 터널 절차를 완료하고 `Policy-R1 (GPU)` 커널을 선택하세요.** Mac의 로컬 커널에서 원격 GPU를 직접 사용할 수는 없습니다. 예제 정책은 모두 가상이며 실제 제도를 설명하지 않습니다.

## 1. 커널과 환경 확인
이 노트북과 `policy_r1` 폴더가 같은 디렉터리에 있어야 합니다. Python 3.10 또는 3.11을 사용합니다.

In [ ]:
# 역할: 현재 커널의 Python과 프로젝트 위치를 확인합니다.
import sys
from pathlib import Path
print("Python:", sys.version)
print("Kernel executable:", sys.executable)
assert (3, 10) <= sys.version_info[:2] <= (3, 11), "Python 3.10/3.11 커널을 선택하세요."
PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "policy_r1").is_dir(), "노트북을 Search-R1 프로젝트 루트에서 실행하세요."
print("Project:", PROJECT_ROOT)

In [ ]:
# 역할: GPU 서버에 설치된 학습 환경과 데이터/검색을 사전 점검합니다.
import subprocess
subprocess.run([sys.executable, "scripts/check_environment.py"], check=True)

설치가 안 되어 있다면 README의 서버 터미널 명령으로 설치하세요. 설치/버전 변경 후에는 커널을 재시작합니다. `CUDA GPU 없음`이면 서버의 커널과 NVIDIA 드라이버를 확인합니다.

## 2. 모델 접근 인증
[모델 페이지](https://huggingface.co/naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-1.5B)에서 이용 조건에 동의한 계정의 Hugging Face read token을 사용하세요. 토큰은 숨김 입력으로 받으며 셀에 직접 적지 않습니다.

In [ ]:
# 역할: Hugging Face 인증이 없으면 토큰을 숨김 입력으로 받아 로그인합니다.
from getpass import getpass
from huggingface_hub import get_token, login
if not get_token():
    login(token=getpass("Hugging Face read token: "), add_to_git_credential=False)
print("인증 설정 완료. 모델 접근 권한은 다음 로딩 단계에서 확인합니다.")

## 3. 실험 설정과 데이터
기존 `outputs/first_experiment`를 보존하고 새 출력 폴더를 사용합니다. 아래 수정 파일은 **GPU 서버 `/root/Search-R1`에도 복사한 뒤 커널을 재시작**해야 반영됩니다. Mac에서 파일만 수정해도 이미 실행 중인 서버 커널의 코드는 바뀌지 않습니다.

기본 데이터는 가상 정책입니다. CLI의 순수 PPO 설정은 그대로이며, 이 노트북에서는 5.1번의 선택적 형식 SFT를 명시적으로 실행합니다.


In [ ]:
# 역할: 원본 결과를 보존하면서 수정된 코드로 새 실험을 설정합니다.
from datetime import datetime
from policy_r1.config import load_config
import policy_r1.trainer as trainer_module
print("사용 중인 학습 코드:", trainer_module.__file__)
assert hasattr(trainer_module.PPOExperiment, "warmup_protocol"), "수정된 policy_r1 파일을 서버에 복사하고 커널을 재시작하세요."
config = load_config("configs/first_experiment.yaml")
config.output_dir = f"outputs/protocol_fix_{datetime.now():%Y%m%d_%H%M%S}"
# 실제 데이터에서는 corpus_path / train_path / val_path / allow_demo를 변경하세요.
# 재개할 때는 checkpoint의 데이터/모델 설정과 일치시켜야 합니다.
config.validate()
config.to_dict()


In [ ]:
# 역할: 데이터 형식·중복·근거 문서 존재 여부 및 BM25 검색 품질을 확인합니다.
from policy_r1.data import load_datasets
from policy_r1.retrieval import build_retriever
corpus, train_rows, val_rows = load_datasets(config)
retriever = build_retriever(config, corpus)
print(f"문서 {len(corpus)}, 학습 질문 {len(train_rows)}, 평가 질문 {len(val_rows)}")
for hit in retriever.search(train_rows[0]["question"], config.top_k):
    print(hit["id"], round(hit["score"], 3), hit["contents"])
# 검색기가 Dense인 경우 모델 중복 로딩을 피하기 위해 점검용 검색기를 해제합니다.
del retriever

## 4. Actor와 독립 Critic 로딩
두 HyperCLOVA backbone을 별도로 로드하고 각각 LoRA를 붙입니다. Critic에는 Value Head를 추가합니다. Reference는 Actor의 고정 base를 공유하며 adapter를 끈 상태로 계산합니다. 첫 다운로드에는 시간이 걸릴 수 있습니다.

In [ ]:
# 역할: 학습 객체를 생성하고 모델·옵티마이저를 GPU에 준비합니다.
from policy_r1.trainer import PPOExperiment
RESUME_FROM = None  # 예: "outputs/first_experiment/checkpoint-00005"
experiment = PPOExperiment(config, resume_from=RESUME_FROM)

## 5. 형식 학습 전 생성과 평가
원하는 동작은 `<search>` → 실제 `<information>` 관측 → `<answer>`입니다.
`assistant -> tool/python`, `search(...)`, 코드블록과 모델이 쓴 가짜 검색 결과는 검색으로 실행하지 않습니다.
`hit_token_limit=True`와 `generation_limit`는 생성 길이 제한에 도달한 경우입니다. 태그가 잘못된 채 192토큰을 다 쓴다면, 길이만 늘리기 전에 형식을 교정해야 합니다.


In [ ]:
# 역할: 학습 전 생성과 검색 trace를 확인합니다.
trajectory = experiment.rollout(val_rows[0]["question"], sample=False)
for event in trajectory.events:
    print(event)
print("종료 사유:", trajectory.finish_reason, "최종 답변:", trajectory.answer)
print("전체 토큰:", len(trajectory.input_ids), "PPO action 토큰:", sum(trajectory.action_mask))

In [ ]:
# 역할: 형식 SFT 이전 모델의 평가를 별도로 보존합니다.
base_before = experiment.evaluate()
base_before


## 5.1. 검색·답변 형식 학습과 PPO 사전 점검
초기 모델이 protocol을 모르면 정답 보상도 0이어서 PPO가 시작되지 않을 수 있습니다.
아래 SFT는 **학습 split의 질문·정답과 실제 검색된 문서만** 사용합니다. 질문/검색 결과 토큰은 loss에서 제외하고 `<search>...</search>`와 `<answer>...</answer>` action을 지도학습합니다. 평가 질문·정답은 사용하지 않습니다.

이는 순수 PPO 이전의 별도 SFT 단계입니다. Reference는 계속 **adapter를 끈 초기 base**이며, warmup 후 KL이 0이 아닐 수 있습니다. 보상은 여전히 정답 EM 기준이며 형식 보상을 자동으로 올리지 않습니다.

새 실험에서 기본 40 update를 수행합니다. 이는 형식 습득을 보장하는 횟수가 아닙니다. 다음 점검에서 계속 invalid이면 trace를 확인하고, PPO를 시작하기 전 이 셀의 warmup을 추가로 실행할 수 있습니다. 체크포인트 재개 시 warmup은 건너뜁니다.


In [ ]:
# 역할: 선택적 train-only SFT를 먼저 실행합니다. PPO update 수와 별도입니다.
RUN_PROTOCOL_WARMUP = True
WARMUP_STEPS = 40
if RUN_PROTOCOL_WARMUP and RESUME_FROM is None:
    warmup_history = experiment.warmup_protocol(steps=WARMUP_STEPS, learning_rate=1e-4)
print("누적 형식 SFT update:", experiment.protocol_warmup_steps)


In [ ]:
# 역할: 실제 sampled rollout을 점검하고 학습 신호 없이 PPO를 시작하지 않도록 합니다.
report = experiment.diagnose_rollouts(count=8, sample=True)
print(report["summary"])
for example in report["examples"]:
    print(example["id"], example["finish_reason"], "reward=", example["reward"])
    for event in example["events"]:
        print(event)
if report["summary"]["valid_answer"] == 0:
    raise RuntimeError("8개 sampled rollout에 유효한 answer가 없습니다. trace의 형식/길이를 먼저 수정하거나, PPO 시작 전 형식 warmup을 보완하세요. 아직 학습 성공이 아닙니다.")
# 이 값과 학습 후 지표를 비교해야 PPO만의 변화를 구분할 수 있습니다.
before = experiment.evaluate()
print("형식 학습 후 / PPO 전:", before)


## 6. PPO 학습
모델이 직접 생성한 rollout으로 학습합니다. SFT의 정답 action을 대신 넣지 않습니다.
`actor_grad_norm`, `critic_grad_norm`, `advantage_std`, `finish_reasons`를 확인하세요. Actor/Critic gradient가 모두 0인 상태가 기본 3회 연속이면 예외를 내고 중단하며, metrics와 **batch 전체** rollout trace를 남깁니다. `actor_loss` 평균만 0인 것은 정상일 수 있으므로 gradient도 함께 확인합니다.


In [ ]:
# 역할: rollout → reference KL/GAE → clipped PPO·Value loss 업데이트를 반복합니다.
history = experiment.train()

## 7. 학습 후 평가와 학습 곡선

In [ ]:
# 역할: base → 형식 SFT → PPO 결과를 구분합니다.
after = experiment.evaluate()
print("초기 모델 / 형식 학습 전:", base_before)
print("형식 학습 후 / PPO 전:", before)
print("PPO 후:", after)
print("형식 SFT update:", experiment.protocol_warmup_steps)
print("저장 경로:", Path(config.output_dir).resolve())


In [ ]:
# 역할: loss와 실제 gradient, 정답률을 함께 확인합니다.
import json
import matplotlib.pyplot as plt
rows = [json.loads(line) for line in (Path(config.output_dir) / "metrics.jsonl").read_text().splitlines()]
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, key in zip(axes.flat, ["actor_loss", "critic_loss", "exact_match", "actor_grad_norm", "critic_grad_norm", "valid_answer"]):
    ax.plot([r["step"] for r in rows], [r[key] for r in rows], marker="o")
    ax.set(title=key, xlabel="PPO update")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# 역할: 학습된 Actor로 원하는 정책 질문에 답하고 검색 근거 trace를 확인합니다.
question = val_rows[0]["question"]
result = experiment.rollout(question, sample=False)
print("답변:", result.answer)
for event in result.events:
    print(event)

## 8. 체크포인트 재개
새 커널에서 1~3번을 실행한 뒤 `config.total_updates`를 저장된 step보다 크게 설정하고, 4번의 `RESUME_FROM`에 checkpoint 경로를 지정하세요. 그 다음 5~7번을 실행하면 이어서 학습합니다. 모델을 이미 로딩한 커널에서 새 객체를 중복 생성하면 메모리가 부족할 수 있습니다.

```python
config.total_updates = 20
RESUME_FROM = "outputs/first_experiment/checkpoint-00010"
experiment = PPOExperiment(config, resume_from=RESUME_FROM)
```

저장된 설정과 달라도 되는 항목은 `total_updates`, `output_dir`, `save_every`, `eval_every`입니다. 기존 checkpoint와 겹치지 않도록 이전 checkpoint에서 분기할 때는 새 `output_dir`를 지정하세요.

**검증 범위:** 제공된 테스트는 CPU의 작은 무작위 Llama로 수식·마스킹·역전파를 확인합니다. 실제 HyperCLOVA 모델과 서버의 CUDA/4-bit 환경은 위 셀에서 별도로 확인해야 합니다. 가상 데이터의 학습 전후 수치는 연구 성능 지표가 아닙니다.